# Memory

# import & 환경변수

In [1]:
from dotenv import load_dotenv
print(load_dotenv())

from langchain_openai.chat_models import ChatOpenAI
from langchain_core.prompts.chat import ChatPromptTemplate
from langchain_core.messages.human import HumanMessage
from langchain_core.messages.ai import AIMessage

True


# 메모리 없는 경우

In [3]:
llm = ChatOpenAI(
    model='gpt-4o',
    temperature=0.1
)

prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful AI talking to a human"),
        ("human", "{question}"),    
])

chain = prompt | llm

def invoke_chain(question):
    result = chain.invoke({"question": question})
    print('🐹', result.content)

invoke_chain("My name is John")
invoke_chain("What is my name?")

🐹 Nice to meet you, John! How can I assist you today?
🐹 I'm sorry, but I don't have access to personal information about you, including your name. If you'd like to share your name or any other details, feel free to do so!


# 메모리 구현 - MessagesPlaceholder

In [6]:
from langchain_core.prompts.chat import MessagesPlaceholder

llm = ChatOpenAI(
    model='gpt-4o',
    temperature=0.1
)

# AI 와 주고 받은 Message 들 저장
chat_history = []

prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful AI talking to a human"),
        # 'history' 라는 키값으로 Message 들을 건네주면 이전에 주고받은 Message 들로 Prompt 만들어준다.
        MessagesPlaceholder(variable_name='history'),
        ("human", "{question}"),    
])

chain = prompt | llm

def invoke_chain(question, history):
    result = chain.invoke({"question": question, "history": chat_history})
    print('🐹', result.content)
    history.append(HumanMessage(content=question))
    history.append(AIMessage(content=result.content))
    
print('🔷 chat_hisotry:', chat_history)
invoke_chain("My name is John", chat_history)
print('🔷 chat_hisotry:', chat_history)
invoke_chain("What is my name?", chat_history)
print('🔷 chat_hisotry:', chat_history)

🔷 chat_hisotry: []
🐹 Nice to meet you, John! How can I assist you today?
🔷 chat_hisotry: [HumanMessage(content='My name is John', additional_kwargs={}, response_metadata={}), AIMessage(content='Nice to meet you, John! How can I assist you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
🐹 Your name is John. How can I help you further?
🔷 chat_hisotry: [HumanMessage(content='My name is John', additional_kwargs={}, response_metadata={}), AIMessage(content='Nice to meet you, John! How can I assist you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}), AIMessage(content='Your name is John. How can I help you further?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


In [ ]:
"""
잘 동작하긴 하나 위 코드의 경우
매번 chain 을 호출할때 chat_history 를 넘겨주어야 한다.
  invoke_chain("My name is Harang", chat_history) <-- !!

자동으로 chat_history  를 건네주려면?  ↓↓↓
"""
None

# 메모리 구현 - RunnablePassthrough

In [7]:
from langchain_core.runnables.passthrough import RunnablePassthrough

In [11]:
llm = ChatOpenAI(
    model='gpt-4o',
    temperature=0.1
)

chat_history = []

prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful AI talking to a human"),
        MessagesPlaceholder(variable_name='history'),
        ("human", "{question}"),    
])

def load_memory(_):
    # print('🎃', _)
    return chat_history

# RunRunnablePassthrough 는 전달받은 key 를 체인의 다음요소까지 전달 (key 값 변경 없이!)
chain = RunnablePassthrough.assign(history=load_memory) | prompt | llm

def invoke_chain(question, history):
    result = chain.invoke({"question": question})  # chain 호출코드에서 chat_history 를 매번 입력하지 않아도 되게 만든다!
    print('🐹', result.content)
    history.append(HumanMessage(content=question))
    history.append(AIMessage(content=result.content))
    
print('🔷 chat_hisotry:', chat_history)
invoke_chain("My name is John", chat_history)
print('🔷 chat_hisotry:', chat_history)
invoke_chain("What is my name?", chat_history)
print('🔷 chat_hisotry:', chat_history)

🔷 chat_hisotry: []
🐹 Nice to meet you, John! How can I assist you today?
🔷 chat_hisotry: [HumanMessage(content='My name is John', additional_kwargs={}, response_metadata={}), AIMessage(content='Nice to meet you, John! How can I assist you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
🐹 Your name is John.
🔷 chat_hisotry: [HumanMessage(content='My name is John', additional_kwargs={}, response_metadata={}), AIMessage(content='Nice to meet you, John! How can I assist you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}), AIMessage(content='Your name is John.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
